# NRGF Implementation: Normalising Radial Graded Filter / NRGF 구현

**Paper**: Huw Morgan, Shadia Rifai Habbal, Richard Woo. *The Depiction of Coronal Structure in White Light Images*. Solar Physics 236, 263-272 (2006).

This notebook implements the NRGF — a one-line algorithm that standardises every annulus of a coronagraph image:

$$
I'(r,\phi) = \frac{I(r,\phi) - \langle I(r)\rangle_\phi}{\sigma(r)_\phi}.
$$

Outline:
1. Build a synthetic 256×256 coronagraph image with a steep radial gradient and embedded structure (streamers, polar plumes, a mock CME).
2. Implement Cartesian → polar coordinate mapping with a sub-pixel-accurate Sun centre.
3. Implement the core NRGF as a per-annulus z-score, including occulter masking.
4. Apply NRGF and visualise structures hidden in the raw image.
5. Compare NRGF against simple alternatives (log stretch, unsharp mask).
6. Demonstrate the analytic identity from the notes (NRGF on a separable model exactly recovers the structure).
7. Summary table mapping the paper's concepts to modern equivalents.

Computational scope: 256×256 image, runs in <5 s on CPU.

본 노트북은 NRGF — 코로나그래프 영상의 각 annulus를 z-score로 표준화하는 한 줄 알고리즘 — 을 구현한다. 256×256 합성 영상, CPU 5초 이내 실행.

In [ ]:
from typing import Tuple

import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

np.random.seed(0)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['font.size'] = 10

## Part 1: Synthetic White-Light Coronagraph Image / 합성 백색광 코로나그래프 영상

Build a 2D image that mimics the dominant features of LASCO C2 white-light data: a steep radial brightness drop, two bright equatorial streamers, several polar plumes, a faint mock CME, and a central occulter mask. The radial drop is exponential in $r$ (a coarse model of the K-corona).

LASCO C2 백색광 영상의 지배적 특징을 합성한다 — 가파른 방사형 감쇠, 두 적도 streamer, 극지 plume, 희미한 mock CME, 중앙 occulter 마스크.

In [ ]:
def synthetic_corona(size: int = 256, occulter_radius: float = 0.18) -> Tuple[np.ndarray, np.ndarray]:
    """Build a synthetic 2D coronagraph image and a validity mask.

    Args:
        size: Image side length in pixels.
        occulter_radius: Radius (fraction of half-image) below which pixels are masked.

    Returns:
        Tuple (image, mask) of shape (size, size). image has values >= 0; mask is 1 outside
        the occulter and 0 inside.
    """
    yy, xx = np.meshgrid(
        np.linspace(-1, 1, size), np.linspace(-1, 1, size), indexing='ij'
    )
    r = np.sqrt(xx ** 2 + yy ** 2)
    phi = np.arctan2(yy, xx)
    radial = np.exp(-3.5 * r) + 5e-4  # ~ 4 decades drop across the field
    streamer = np.exp(-((phi) ** 2) / 0.04) + np.exp(-((phi - np.pi) ** 2) / 0.04)
    plumes = (
        np.exp(-(((phi - np.pi / 2) ** 2) / 0.001))
        + np.exp(-(((phi + np.pi / 2) ** 2) / 0.001))
    ) * np.exp(-2.0 * r)
    mock_cme = (
        np.exp(-(((phi - 0.7) ** 2) / 0.02))
        * np.exp(-((r - 0.55) ** 2) / 0.005)
    )
    structure = 1.0 + 0.7 * streamer + 0.4 * plumes + 0.6 * mock_cme
    img = radial * structure
    mask = (r >= occulter_radius).astype(np.float64)
    img = img * mask
    img = img + 0.005 * np.sqrt(np.maximum(img, 1e-6)) * np.random.randn(size, size)
    return img, mask


image, mask = synthetic_corona(size=256, occulter_radius=0.18)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image, cmap='gray'); axes[0].set_title('Raw (linear)'); axes[0].axis('off')
axes[1].imshow(np.log10(image + 1e-4), cmap='gray')
axes[1].set_title('Raw (log10)'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## Part 2: Sun Centre and Per-Pixel Polar Coordinates / 태양 중심과 픽셀별 극좌표

Compute heliocentric distance and position angle for every pixel, given a sub-pixel-accurate Sun centre. Here the synthetic image has the Sun at the geometric centre, but the function accepts an arbitrary centre for real data.

주어진 sub-pixel 태양 중심에 대해 각 픽셀의 heliocentric 거리와 위치각을 계산한다. 합성 영상은 기하학적 중심을 사용하지만, 함수는 임의 중심을 받는다.

In [ ]:
def polar_coordinates(
    shape: Tuple[int, int], centre: Tuple[float, float]
) -> Tuple[np.ndarray, np.ndarray]:
    """Compute per-pixel radius and azimuth around a given centre.

    Args:
        shape: Tuple (Ny, Nx).
        centre: Sub-pixel-accurate centre coordinate (cy, cx) in pixel units.

    Returns:
        Tuple (r, phi) of arrays of shape (Ny, Nx).
        r is radius in pixels, phi is azimuth in radians in [-pi, pi).
    """
    Ny, Nx = shape
    cy, cx = centre
    yy, xx = np.meshgrid(
        np.arange(Ny) - cy, np.arange(Nx) - cx, indexing='ij'
    )
    r = np.sqrt(yy ** 2 + xx ** 2)
    phi = np.arctan2(yy, xx)
    return r, phi


size = image.shape[0]
r_pix, phi_pix = polar_coordinates(image.shape, centre=((size - 1) / 2.0, (size - 1) / 2.0))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(r_pix, cmap='viridis'); axes[0].set_title('r (pixels)'); axes[0].axis('off')
axes[1].imshow(phi_pix, cmap='twilight'); axes[1].set_title('phi (rad)'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## Part 3: Core NRGF Implementation / 핵심 NRGF 구현

Bin pixels by radius into $N_r$ annuli, compute the mean and stdev of valid pixels in each annulus, then standardise. Pixels outside the mask (occulter, edges) are excluded from the statistics and set to NaN in the output.

픽셀을 $N_r$ 개의 annulus로 묶어, 각 annulus에서 유효 픽셀의 평균과 표준편차를 계산하고 표준화한다. 마스크 밖(occulter, 경계) 픽셀은 통계에서 제외되고 출력에서 NaN으로 표시된다.

In [ ]:
def nrgf(
    image: np.ndarray,
    mask: np.ndarray,
    centre: Tuple[float, float],
    n_radial_bins: int = 200,
    r_min: float = 0.0,
    r_max: float = None,
) -> np.ndarray:
    """Apply the Normalising Radial Graded Filter (Morgan et al. 2006, Eq. 1).

    Args:
        image: 2D float array.
        mask: 2D float/bool array of the same shape; 1 for valid, 0 for invalid.
        centre: (cy, cx) sub-pixel-accurate Sun centre.
        n_radial_bins: Number of radial bins (annuli).
        r_min: Inner radius (pixels). Pixels with r < r_min are excluded.
        r_max: Outer radius (pixels). If None, uses the largest valid radius.

    Returns:
        2D float array of NRGF-processed values; NaN where invalid.
    """
    r, _ = polar_coordinates(image.shape, centre)
    if r_max is None:
        r_max = float(r[mask > 0].max())
    bin_edges = np.linspace(r_min, r_max, n_radial_bins + 1)
    bin_idx = np.digitize(r, bin_edges) - 1

    out = np.full_like(image, np.nan, dtype=np.float64)
    for k in range(n_radial_bins):
        sel = (bin_idx == k) & (mask > 0)
        if sel.sum() < 8:
            continue
        vals = image[sel]
        mu = vals.mean()
        sd = vals.std()
        if sd < 1e-12:
            continue
        out[sel] = (vals - mu) / sd
    return out


centre = ((image.shape[0] - 1) / 2.0, (image.shape[1] - 1) / 2.0)
nrgf_image = nrgf(image, mask, centre=centre, n_radial_bins=200)
print(f'NRGF output value range: [{np.nanmin(nrgf_image):.2f}, {np.nanmax(nrgf_image):.2f}]')
print(f'mean: {np.nanmean(nrgf_image):.3f}, stdev: {np.nanstd(nrgf_image):.3f}')

## Part 4: Visualise Raw vs NRGF / 원본 vs NRGF 비교

Show the raw image (linear and log stretches) alongside the NRGF output. Structure that is invisible in the raw linear view, and only barely visible in the log view, is clearly seen in the NRGF view at uniform contrast across all heights.

원본(선형, 로그 스트레치)과 NRGF 출력을 나란히 표시한다. 원본 선형 보기에서 안 보이고 로그 보기에서도 희미한 구조가 NRGF에서는 모든 높이에서 *균일한 대비*로 또렷이 보인다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(image, cmap='gray'); axes[0].set_title('Raw (linear)'); axes[0].axis('off')
axes[1].imshow(np.log10(image + 1e-4), cmap='gray')
axes[1].set_title('Raw (log10)'); axes[1].axis('off')
axes[2].imshow(nrgf_image, cmap='gray', vmin=-3, vmax=3)
axes[2].set_title('NRGF (clipped to ±3σ)'); axes[2].axis('off')
plt.tight_layout(); plt.show()

## Part 5: Comparison with Simpler Alternatives / 단순 대안과의 비교

Compare the NRGF output with two simpler enhancement approaches that are commonly used:
1. **Log stretch** — display $\log_{10}(I + \epsilon)$.
2. **Unsharp mask** — subtract a Gaussian-blurred version: $I - G_\sigma * I$.

Both fail in different ways: log stretch still has the radial gradient; unsharp mask emphasises sharp edges but does not equalise the radial dynamic range.

단순 대안 두 가지 — 로그 스트레치와 unsharp mask — 와 비교한다. 로그는 방사형 기울기가 남고, unsharp는 sharp edge만 강조하고 dynamic range는 평탄화 못함.

In [ ]:
def unsharp_mask(image: np.ndarray, sigma: float = 5.0) -> np.ndarray:
    """Classical unsharp masking by Gaussian blur subtraction."""
    return image - gaussian_filter(image, sigma=sigma)


log_image = np.log10(image + 1e-4)
unsharp_image = unsharp_mask(image, sigma=4.0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, im, title in zip(
    axes,
    [image, log_image, unsharp_image, nrgf_image],
    ['Raw (linear)', 'Log stretch', 'Unsharp mask', 'NRGF (±3σ)'],
):
    if title == 'NRGF (±3σ)':
        ax.imshow(im, cmap='gray', vmin=-3, vmax=3)
    else:
        ax.imshow(im, cmap='gray')
    ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()

## Part 6: Analytic Identity Check / 분석적 항등성 확인

From the notes (Part IV / Sec. 7), if the image is exactly separable as $I(r,\phi) = a(r) + b(r) \cdot s(\phi)$ with $\langle s\rangle = 0$, then NRGF must produce $I'(r,\phi) = s(\phi)/\sqrt{\langle s^2\rangle}$ — independent of $r$ and exactly the structure pattern. Verify this on a noiseless synthetic case.

노트(Part IV / Sec. 7)에 따라, 영상이 정확히 $I(r,\phi) = a(r) + b(r)\cdot s(\phi)$ 형태로 분리 가능하고 $\langle s\rangle=0$이면, NRGF 출력은 $r$에 무관하게 정확히 $s(\phi)/\sqrt{\langle s^2\rangle}$이다. 무잡음 합성 영상에서 이 항등성을 검증한다.

In [ ]:
size = 256
yy, xx = np.meshgrid(np.linspace(-1, 1, size), np.linspace(-1, 1, size), indexing='ij')
r_arr = np.sqrt(xx ** 2 + yy ** 2)
phi_arr = np.arctan2(yy, xx)
a_r = np.exp(-3.0 * r_arr)
b_r = 0.3 * a_r
s_phi = np.cos(2 * phi_arr)  # mean-zero pattern
I_separable = a_r + b_r * s_phi
mask_sep = np.ones_like(I_separable)
mask_sep[r_arr < 0.05] = 0  # tiny occulter

centre_sep = ((size - 1) / 2.0, (size - 1) / 2.0)
I_nrgf = nrgf(I_separable, mask_sep, centre_sep, n_radial_bins=200)
expected = s_phi / np.sqrt(np.mean(s_phi ** 2))
valid = ~np.isnan(I_nrgf)
diff = I_nrgf[valid] - expected[valid]
print(f'NRGF vs analytic-expected mean abs error: {np.abs(diff).mean():.4f}')
print(f'NRGF vs analytic-expected max abs error:  {np.abs(diff).max():.4f}')

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(I_separable, cmap='gray'); axes[0].set_title('I(r, phi)'); axes[0].axis('off')
axes[1].imshow(I_nrgf, cmap='gray', vmin=-2, vmax=2)
axes[1].set_title('NRGF output'); axes[1].axis('off')
axes[2].imshow(expected, cmap='gray', vmin=-2, vmax=2)
axes[2].set_title('Analytic expectation s(phi)/√⟨s²⟩'); axes[2].axis('off')
plt.tight_layout(); plt.show()

## Summary / 요약

The notebook demonstrates that the Normalising Radial Graded Filter — a single equation — exposes coronal structure that is invisible in linear/log stretches and superior to classical unsharp masking. The analytic check on a separable model confirms that NRGF *exactly* removes the radial component and recovers the azimuthal structure pattern.

본 노트북은 NRGF가 한 줄의 식만으로 선형/로그 스트레치에서 안 보이는 코로나 구조를 드러내며 고전적 unsharp mask보다 우수함을 보인다. 분리 가능 모델에서의 분석적 검증은 NRGF가 방사형 성분을 *정확히* 제거하고 방위각 구조 패턴을 회복함을 확인한다.

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Per-region statistical normalisation | NRGF: per-annulus z-score | InstanceNorm / GroupNorm in deep networks |
| Background removal | Long-term unpolarized background subtraction | Learned background estimation networks |
| Display dynamic-range compression | $(I-\mu_r)/\sigma_r$ | HDR tone-mapping operators |
| Multiscale structure visualisation | Single-scale annulus normalisation | MGN (Morgan & Druckmüller 2014), MSGN |
| Coordinate-aware preprocessing | Cartesian → polar resampling | Polar-equivariant CNNs, log-polar networks |